In [1]:
# Setup and configuration
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Define the plot saving helper as requested
def save_plot(fig, name, stage):
    folder = f"../outputs/plots/{stage}"
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{name}.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved: {path}")



In [2]:
# Load the raw dataset
raw_path = '../data/raw/merged_city_crime_data.csv'
if not os.path.exists(raw_path):
    # Fallback to local path if running outside notebooks folder
    raw_path = 'data/raw/merged_city_crime_data.csv'

print(f"Loading raw data from: {raw_path}")
df = pd.read_csv(raw_path)

# Display shape and columns
print(f"Raw dataset shape: {df.shape}")
print("\n--- Columns and Types ---")
print(df.info())
print("\n--- First 5 rows ---")
print(df.head())

Loading raw data from: ../data/raw/merged_city_crime_data.csv


Raw dataset shape: (250000, 24)

--- Columns and Types ---


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 24 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              250000 non-null  int64  
 1   State                           250000 non-null  object 
 2   City                            250000 non-null  object 
 3   Locality                        250000 non-null  object 
 4   Property_Type                   250000 non-null  object 
 5   BHK                             250000 non-null  int64  
 6   Size_in_SqFt                    250000 non-null  int64  
 7   Price_in_Lakhs                  250000 non-null  float64
 8   Price_per_SqFt                  250000 non-null  float64
 9   Year_Built                      250000 non-null  int64  
 10  Furnished_Status                250000 non-null  object 
 11  Floor_No                        250000 non-null  int64  
 12  Total_Floors    

## Step 2: Analyze Missing Values

We will check for any missing values across all columns. If missing values are found, we will apply the appropriate imputation strategy (median for numeric, mode or 'Unknown' for categorical).

In [3]:
# Check missing values
missing_counts = df.isnull().sum()
print("Missing values per column:")
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "No missing values found in the dataset!")

Missing values per column:
No missing values found in the dataset!


## Step 3: Data Validation & Floor Swapping

A critical validation check is verifying that the property's floor number (`Floor_No`) is less than or equal to the total number of floors in the building (`Total_Floors`). Let's check how many rows violate this sanity check.

In [4]:
# Check for floor number violations
violations = df[df['Floor_No'] > df['Total_Floors']]
print(f"Number of rows where Floor_No > Total_Floors: {len(violations)} ({len(violations)/len(df)*100:.2f}%)")
print("\nSample violations:")
print(violations[['Floor_No', 'Total_Floors']].head(10))

Number of rows where Floor_No > Total_Floors: 116304 (46.52%)

Sample violations:
    Floor_No  Total_Floors
0         22             1
1         21            20
4          3             2
5         27             1
6         16             5
7         24             4
8         23            16
12         9             1
13        13             4
14        26            18


### Floor Swapping Strategy

Since nearly 46.5% of the rows have `Floor_No > Total_Floors`, dropping them would lead to severe data loss. This issue is highly typical of swapped data columns. We will resolve this by swapping `Floor_No` and `Total_Floors` for these rows. This is mathematically implemented as:
- `Floor_No = min(Floor_No, Total_Floors)`
- `Total_Floors = max(Floor_No, Total_Floors)`

In [5]:
# Apply floor swapping logic
df_swapped = df.copy()
temp_floor = df_swapped['Floor_No'].copy()
mask = df_swapped['Floor_No'] > df_swapped['Total_Floors']

df_swapped.loc[mask, 'Floor_No'] = df_swapped.loc[mask, 'Total_Floors']
df_swapped.loc[mask, 'Total_Floors'] = temp_floor[mask]

# Verify no violations remain
remaining_violations = (df_swapped['Floor_No'] > df_swapped['Total_Floors']).sum()
print(f"Remaining floor violations after swapping: {remaining_violations}")
df = df_swapped

Remaining floor violations after swapping: 0


## Step 4: Outlier Detection & Capping

We will examine key numeric columns (`Price_in_Lakhs`, `Size_in_SqFt`, and `Crime_Rate_Per_Lakh`) for outliers. We will visualize the distributions using boxplots and save them. To preserve the dataset size (250,000 rows) while preventing extreme values from distorting models, we will **cap** outliers at the 1.5 * IQR bounds.

In [6]:
# Outlier detection helper
outlier_cols = ['Price_in_Lakhs', 'Size_in_SqFt', 'Crime_Rate_Per_Lakh']
capping_bounds = {}

for col in outlier_cols:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(y=df[col], ax=ax, color='skyblue')
    ax.set_title(f"Boxplot of {col} (Before Capping)")
    
    # Calculate IQR and bounds
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    capping_bounds[col] = (lower_bound, upper_bound)
    
    # Count outliers
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col}: Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}, Bounds=({lower_bound:.2f}, {upper_bound:.2f})")
    print(f"  Number of outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
    
    save_plot(fig, f"{col.lower()}_before_capping_boxplot", "cleaning")
    plt.close(fig)

Price_in_Lakhs: Q1=160.71, Q3=303.39, IQR=142.68, Bounds=(-53.31, 517.41)
  Number of outliers: 6783 (2.71%)


Saved: ../outputs/plots/cleaning/price_in_lakhs_before_capping_boxplot.png


Size_in_SqFt: Q1=1623.00, Q3=3874.00, IQR=2251.00, Bounds=(-1753.50, 7250.50)
  Number of outliers: 0 (0.00%)


Saved: ../outputs/plots/cleaning/size_in_sqft_before_capping_boxplot.png


Crime_Rate_Per_Lakh: Q1=291.30, Q3=570.30, IQR=279.00, Bounds=(-127.20, 988.80)
  Number of outliers: 31208 (12.48%)


Saved: ../outputs/plots/cleaning/crime_rate_per_lakh_before_capping_boxplot.png


### Capping Outliers

We will cap the values outside the 1.5 * IQR range. Values below the lower bound will be set to the lower bound, and values above the upper bound will be set to the upper bound. Since the minimum price and size are already well within bounds, this will primarily affect high-end prices and high crime rate areas.

In [7]:
# Apply capping
df_capped = df.copy()
for col in outlier_cols:
    lower_bound, upper_bound = capping_bounds[col]
    # For housing price and size, let's keep lower bound at actual minimums if lower bound is negative
    actual_lower = max(lower_bound, df[col].min())
    df_capped[col] = np.clip(df_capped[col], actual_lower, upper_bound)
    print(f"Capped {col} to range: [{actual_lower:.2f}, {upper_bound:.2f}]")

# Verify with new boxplots
for col in outlier_cols:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(y=df_capped[col], ax=ax, color='lightgreen')
    ax.set_title(f"Boxplot of {col} (After Capping)")
    save_plot(fig, f"{col.lower()}_after_capping_boxplot", "cleaning")
    plt.close(fig)

df = df_capped

Capped Price_in_Lakhs to range: [38.30, 517.41]
Capped Size_in_SqFt to range: [500.00, 7250.50]
Capped Crime_Rate_Per_Lakh to range: [83.90, 988.80]


Saved: ../outputs/plots/cleaning/price_in_lakhs_after_capping_boxplot.png


Saved: ../outputs/plots/cleaning/size_in_sqft_after_capping_boxplot.png


Saved: ../outputs/plots/cleaning/crime_rate_per_lakh_after_capping_boxplot.png


## Step 5: Sanity Checks & Standardizing Formats

We will verify that:
- `Price_in_Lakhs` > 0
- `Size_in_SqFt` > 0
- `BHK` is between 1 and 10
We will also standardize the text columns (consistent title casing, strip whitespaces) for `State`, `City`, `Locality`, `Property_Type`, and `Furnished_Status`.

In [8]:
# Sanity checks
print("Sanity Check: Minimum Price > 0:", (df['Price_in_Lakhs'] > 0).all())
print("Sanity Check: Minimum Size > 0:", (df['Size_in_SqFt'] > 0).all())
print("Sanity Check: BHK between 1-10:", ((df['BHK'] >= 1) & (df['BHK'] <= 10)).all())
print("Sanity Check: Floor Number <= Total Floors:", (df['Floor_No'] <= df['Total_Floors']).all())

# Standardize text formats
text_cols = ['State', 'City', 'Locality', 'Property_Type', 'Furnished_Status', 
             'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Facing', 'Owner_Type', 'Availability_Status']

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    # Fix specific casing like 'Semi-Furnished' if it becomes 'Semi-Furnished'
    df[col] = df[col].replace('Semi-Furnished', 'Semi-furnished')

print("\nCategorical casing standardized.")

Sanity Check: Minimum Price > 0: True
Sanity Check: Minimum Size > 0: True
Sanity Check: BHK between 1-10: True
Sanity Check: Floor Number <= Total Floors: True



Categorical casing standardized.


## Step 6: Duplicate Detection & Removal

We will check for exact duplicate rows or near-duplicates (matching City, Locality, Size, Price, BHK, Year_Built, Floor_No) and drop them.

In [9]:
# Check for exact duplicates
exact_dups = df.duplicated().sum()
print(f"Exact duplicates found: {exact_dups}")

# Check for near-duplicates (excluding ID)
near_dup_cols = ['State', 'City', 'Locality', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Year_Built', 'Floor_No', 'Total_Floors']
near_dups = df.duplicated(subset=near_dup_cols).sum()
print(f"Near-duplicates (matching main features) found: {near_dups}")

if near_dups > 0:
    df = df.drop_duplicates(subset=near_dup_cols, keep='first')
    print(f"Dropped {near_dups} near-duplicate rows. New shape: {df.shape}")

Exact duplicates found: 0


Near-duplicates (matching main features) found: 0


## Step 7: Save Cleaned Dataset

We will save the cleaned, validated, and outlier-capped dataset to `data/interim/cleaned_dataset.csv`. As confirmation, we will print the shape and dtypes.

In [10]:
# Save cleaned dataset
output_path = '../data/interim/cleaned_dataset.csv'
if not os.path.exists('../data/interim'):
    # Fallback to local path if running outside notebooks folder
    output_path = 'data/interim/cleaned_dataset.csv'
    os.makedirs('data/interim', exist_ok=True)

df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")
print(f"Final shape: {df.shape}")
print("\n--- Final Datatypes ---")
print(df.dtypes)

Cleaned dataset saved to: ../data/interim/cleaned_dataset.csv
Final shape: (250000, 24)

--- Final Datatypes ---
ID                                  int64
State                              object
City                               object
Locality                           object
Property_Type                      object
BHK                                 int64
Size_in_SqFt                        int64
Price_in_Lakhs                    float64
Price_per_SqFt                    float64
Year_Built                          int64
Furnished_Status                   object
Floor_No                            int64
Total_Floors                        int64
Age_of_Property                     int64
Nearby_Schools                      int64
Nearby_Hospitals                    int64
Public_Transport_Accessibility     object
Parking_Space                      object
Security                           object
Amenities                          object
Facing                             object
Owner